# BigTest — an input per path through the query

Rather than searching for an input that happens to reach a branch, read the query's own
conditions, solve them, and construct a record that takes it.

In [ ]:
import os, sys, glob

ROOT = os.environ.get("BIGASTERISK_HOME") or os.path.abspath("..")

# Jars: a source checkout has them under modules/*/target, the Docker image under jars/.
JARS = sorted(glob.glob(f"{ROOT}/modules/*/target/scala-2.13/bigasterisk-*.jar")) \
    or sorted(glob.glob(f"{ROOT}/jars/bigasterisk-*.jar"))
if not JARS:
    raise SystemExit("No BigAsterisk jars found. Run: bin/sbt package")

FASTUTIL_JAR = os.environ.get("FASTUTIL_JAR") or next(iter(sorted(
    glob.glob(f"{ROOT}/jars/fastutil*.jar")
    + glob.glob(os.path.expanduser("~/Library/Caches/Coursier/**/fastutil-8.5.15.jar"), recursive=True)
    + glob.glob(os.path.expanduser("~/.cache/coursier/**/fastutil-8.5.15.jar"), recursive=True)
)), None)
if not FASTUTIL_JAR:
    raise SystemExit("fastutil jar not found. Run: bin/sbt package")

SPARK_JARS = ",".join(JARS + [FASTUTIL_JAR])
DATA = f"{ROOT}/examples/data"
sys.path.insert(0, f"{ROOT}/python")

## The data

Twelve orders across three customers. One of them, `o8`, is an outlier at
`99999` — every notebook here uses it as the thing to find.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import bigasterisk

spark = (bigasterisk.configure(SparkSession.builder)
    .master("local[2]")
    .appName("bigtest-notebook")
    .config("spark.jars", SPARK_JARS)
    .config("spark.sql.adaptive.skewJoin.enabled", "false")
    .config("spark.ui.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

orders = spark.read.schema("oid STRING, cid STRING, amount INT").csv(f"{DATA}/orders.txt")
customers = spark.read.schema("cid STRING, name STRING").csv(f"{DATA}/customers.txt")
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

orders.show()

## Generate

Every generated input is executed and the branch it was built for is checked, so `verified` reports what happened rather than what was intended.

In [ ]:
suite = bigasterisk.testgen(spark).generate(
    "SELECT cid FROM orders WHERE amount > 100",
    {"orders": orders}, rows_per_path=1, natural=False)

for case in suite.cases:
    print(case)

## Unsolvable paths are reported, not fabricated

In [ ]:
impossible = bigasterisk.testgen(spark).generate(
    "SELECT cid FROM orders WHERE amount > 200 AND amount < 100",
    {"orders": orders})
for case in impossible.cases:
    print(case.path, "->", case.note)

## Check

`> 100` on an integer column means the boundary is 101.

In [ ]:
taking = [c for c in suite.cases if not c.path.startswith("NOT")][0]
assert taking.verified, taking.note
assert "101" in taking.tables["orders"][0], taking.tables
assert any(c.note == "unsatisfiable" for c in impossible.cases)
print("OK")